# 📈 SahamMeter — Analisis Big Data Saham IDX

**Mata Kuliah:** Big Data dan Data Lakehouse  
**ETS Praktik:** Big Data Pipeline End-to-End  
**Komponen 3:** Apache Spark — Processing Layer

Notebook ini membaca data saham blue-chip IDX yang sudah tersimpan di HDFS,
lalu melakukan 3 analisis wajib:
1. **Return per saham** — saham mana yang naik/turun paling banyak?
2. **Volatilitas intraday** — saham mana yang paling fluktuatif?
3. **Frekuensi sebutan di berita** — perusahaan mana yang paling banyak disebut di RSS?

Hasil disimpan kembali ke HDFS dan ke `dashboard/data/spark_results.json`.

## 0. Setup & Inisialisasi SparkSession

In [1]:
import os
import json
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.window import Window

# ── Konfigurasi path ─────────────────────────────────────────────────────────
HDFS_NAMENODE   = "hdfs://namenode:8020"
HDFS_API_PATH   = f"{HDFS_NAMENODE}/data/saham/api/"
HDFS_RSS_PATH   = f"{HDFS_NAMENODE}/data/saham/rss/"
HDFS_HASIL_PATH = f"{HDFS_NAMENODE}/data/saham/hasil/"

# Dashboard: folder di-mount ke /home/jovyan/dashboard di dalam container
DASHBOARD_DATA_DIR = "/home/jovyan/dashboard/data"
os.makedirs(DASHBOARD_DATA_DIR, exist_ok=True)
SPARK_RESULTS_PATH = os.path.join(DASHBOARD_DATA_DIR, "spark_results.json")

print("✅ Library berhasil diimport")
print(f"📁 Dashboard data dir : {DASHBOARD_DATA_DIR}")

✅ Library berhasil diimport
📁 Dashboard data dir : /home/jovyan/dashboard/data


In [2]:
spark = (
    SparkSession.builder
    .appName("SahamMeter-Analysis")
    .config("spark.hadoop.fs.defaultFS", HDFS_NAMENODE)
    .config("spark.sql.shuffle.partitions", "4")
    # Jalankan sebagai root agar bisa write ke HDFS
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .getOrCreate()
)

# Override HDFS user menjadi root supaya punya izin write
os.environ["HADOOP_USER_NAME"] = "root"

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession aktif : {spark.version}")
print(f"🌐 HDFS Namenode      : {HDFS_NAMENODE}")
print(f"👤 HADOOP_USER_NAME   : {os.environ.get('HADOOP_USER_NAME')}")

✅ SparkSession aktif : 3.3.0
🌐 HDFS Namenode      : hdfs://namenode:8020
👤 HADOOP_USER_NAME   : root


---
## 1. Load Data dari HDFS

In [3]:
# Schema sesuai format producer_api.py
# {"symbol": "BBCA", "price": 5850.0, "currency": "IDR",
#  "volume": 0, "timestamp": "2026-05-03T01:39:14+00:00", "source": "yfinance"}

schema_api = StructType([
    StructField("symbol",    StringType(), True),
    StructField("price",     DoubleType(), True),
    StructField("currency",  StringType(), True),
    StructField("volume",    LongType(),   True),
    StructField("timestamp", StringType(), True),
    StructField("source",    StringType(), True),
])

df_api_raw = (
    spark.read
    .schema(schema_api)
    .json(HDFS_API_PATH)
)

df_api = (
    df_api_raw
    .withColumn("ts",     F.to_timestamp("timestamp"))
    .withColumn("ts_wib", F.from_utc_timestamp(F.col("ts"), "Asia/Jakarta"))
    .withColumn("date",   F.to_date("ts_wib"))
    .withColumn("hour",   F.hour("ts_wib"))
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    .filter(F.col("symbol").isNotNull())
)

total_api = df_api.count()
print(f"📊 Total record saham (API) : {total_api:,}")
print(f"📅 Rentang tanggal : {df_api.agg(F.min('date'), F.max('date')).collect()[0]}")
print()
df_api.show(5, truncate=False)

📊 Total record saham (API) : 375
📅 Rentang tanggal : Row(min(date)=datetime.date(2026, 5, 3), max(date)=datetime.date(2026, 5, 3))

+------+------+--------+------+--------------------------------+--------+--------------------------+--------------------------+----------+----+
|symbol|price |currency|volume|timestamp                       |source  |ts                        |ts_wib                    |date      |hour|
+------+------+--------+------+--------------------------------+--------+--------------------------+--------------------------+----------+----+
|BBCA  |5850.0|IDR     |0     |2026-05-03T01:15:58.531647+00:00|yfinance|2026-05-03 01:15:58.531647|2026-05-03 08:15:58.531647|2026-05-03|8   |
|TLKM  |2810.0|IDR     |0     |2026-05-03T01:15:58.797445+00:00|yfinance|2026-05-03 01:15:58.797445|2026-05-03 08:15:58.797445|2026-05-03|8   |
|ASII  |5975.0|IDR     |0     |2026-05-03T01:15:58.931700+00:00|yfinance|2026-05-03 01:15:58.9317  |2026-05-03 08:15:58.9317  |2026-05-03|8   |
|BBR

In [4]:
# Schema SESUAI format aktual RSS (field: hash, link, published, summary, title)
schema_rss = StructType([
    StructField("hash",      StringType(), True),
    StructField("link",      StringType(), True),
    StructField("published", StringType(), True),
    StructField("summary",   StringType(), True),
    StructField("title",     StringType(), True),
])

df_rss = (
    spark.read
    .option("mode", "PERMISSIVE")
    .schema(schema_rss)
    .json(HDFS_RSS_PATH)
    .filter(F.col("title").isNotNull())
    # Bersihkan HTML tag dari summary untuk analisis teks
    .withColumn("summary_clean", F.regexp_replace(F.col("summary"), "<[^>]+>", ""))
)

total_rss = df_rss.count()
print(f"📰 Total artikel RSS : {total_rss:,}")
df_rss.select("title", "published", "summary_clean").show(5, truncate=60)

📰 Total artikel RSS : 3
+------------------------------------------------------------+-------------------------------+------------------------------------------------------------+
|                                                       title|                      published|                                               summary_clean|
+------------------------------------------------------------+-------------------------------+------------------------------------------------------------+
|    Disrupsi AI: Bagaimana AI Mengubah Lanskap Tenaga Kerja?|Sun, 03 May 2026 09:00:13 +0700| Disrupsi AI memperketat persaingan dan menggeser pekerja...|
|Wajah Lain Pasar Kerja Global dan Tantangan Kualitas Peke...|Sun, 03 May 2026 09:00:36 +0700| Turunnya angka pengangguran terbuka secara global tidak ...|
|Transmart Full Day Sale, Cek Aneka Produk yang Diskon Har...|Sun, 03 May 2026 08:30:35 +0700| Diskon hingga 50% + 20% Transmart Full Day Sale berlaku ...|
+---------------------------------------

In [5]:
# ── Deteksi apakah data flat (bursa tutup / weekend) ─────────────────────────
harga_unik = df_api.groupBy("symbol").agg(
    F.countDistinct("price").alias("harga_unik"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.count("*").alias("total_record")
)
harga_unik.show()

# Cek apakah semua saham punya hanya 1 harga unik (flat)
max_variasi = harga_unik.agg(F.max("harga_unik")).collect()[0][0]
IS_FLAT = (max_variasi == 1)

if IS_FLAT:
    print("⚠️  DATA FLAT terdeteksi — kemungkinan hari libur/weekend (bursa IDX tutup)")
    print("   Analisis tetap dijalankan dengan data yang tersedia.")
    print("   Harga yang tercatat adalah harga penutupan terakhir dari yfinance.")
else:
    print("✅ Data bervariasi — bursa aktif, analisis berjalan normal.")

+------+----------+---------+---------+------------+
|symbol|harga_unik|min_price|max_price|total_record|
+------+----------+---------+---------+------------+
|  BBRI|         1|   2990.0|   2990.0|          75|
|  BBCA|         1|   5850.0|   5850.0|          75|
|  BMRI|         1|   4390.0|   4390.0|          75|
|  ASII|         1|   5975.0|   5975.0|          75|
|  TLKM|         1|   2810.0|   2810.0|          75|
+------+----------+---------+---------+------------+

⚠️  DATA FLAT terdeteksi — kemungkinan hari libur/weekend (bursa IDX tutup)
   Analisis tetap dijalankan dengan data yang tersedia.
   Harga yang tercatat adalah harga penutupan terakhir dari yfinance.


---
## Analisis 1 — Return Per Saham

> **Pertanyaan bisnis:** Saham blue-chip mana yang naik atau turun paling banyak?
>
> **Formula:** `return (%) = (harga_terkini - harga_awal) / harga_awal × 100`
>
> **Catatan:** Jika bursa sedang tutup (weekend/libur), harga tidak bergerak sehingga return = 0%. Analisis ini tetap menampilkan posisi harga terakhir tiap saham sebagai referensi valuasi.

In [6]:
df_api.createOrReplaceTempView("saham")

df_return = spark.sql("""
    WITH ranked AS (
        SELECT
            symbol, price, ts_wib,
            ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY ts_wib ASC)  AS rn_awal,
            ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY ts_wib DESC) AS rn_akhir
        FROM saham
    ),
    harga_awal AS (
        SELECT symbol, price AS harga_awal, ts_wib AS waktu_awal
        FROM ranked WHERE rn_awal = 1
    ),
    harga_akhir AS (
        SELECT symbol, price AS harga_akhir, ts_wib AS waktu_akhir
        FROM ranked WHERE rn_akhir = 1
    )
    SELECT
        a.symbol,
        a.harga_awal,
        b.harga_akhir,
        ROUND((b.harga_akhir - a.harga_awal) / a.harga_awal * 100, 4) AS return_pct,
        ROUND(b.harga_akhir - a.harga_awal, 2)                         AS selisih_rp,
        a.waktu_awal,
        b.waktu_akhir
    FROM harga_awal a
    JOIN harga_akhir b ON a.symbol = b.symbol
    ORDER BY harga_akhir DESC
""")

df_return.show(truncate=False)
df_return.createOrReplaceTempView("hasil_return")
rows_return = df_return.collect()

+------+----------+-----------+----------+----------+--------------------------+--------------------------+
|symbol|harga_awal|harga_akhir|return_pct|selisih_rp|waktu_awal                |waktu_akhir               |
+------+----------+-----------+----------+----------+--------------------------+--------------------------+
|ASII  |5975.0    |5975.0     |0.0       |0.0       |2026-05-03 08:15:58.9317  |2026-05-03 09:30:52.773903|
|BBCA  |5850.0    |5850.0     |0.0       |0.0       |2026-05-03 08:15:58.531647|2026-05-03 09:30:52.385136|
|BMRI  |4390.0    |4390.0     |0.0       |0.0       |2026-05-03 08:15:59.070103|2026-05-03 09:30:52.898169|
|BBRI  |2990.0    |2990.0     |0.0       |0.0       |2026-05-03 08:15:58.661821|2026-05-03 09:30:52.523202|
|TLKM  |2810.0    |2810.0     |0.0       |0.0       |2026-05-03 08:15:58.797445|2026-05-03 09:30:52.650838|
+------+----------+-----------+----------+----------+--------------------------+--------------------------+



In [7]:
print("=" * 65)
print("📈 INTERPRETASI: Return Per Saham")
print("=" * 65)

for row in rows_return:
    if row.return_pct > 0:
        arah = "🟢 NAIK"
    elif row.return_pct < 0:
        arah = "🔴 TURUN"
    else:
        arah = "⚪ FLAT"
    tanda = "+" if row.return_pct >= 0 else ""
    print(f"  {row.symbol:<6} | Rp {row.harga_akhir:>8,.0f} | {tanda}{row.return_pct:.2f}% | {arah}")

print()

if IS_FLAT:
    # Urutkan berdasarkan harga absolut untuk insight yang tetap bermakna
    rows_by_price = sorted(rows_return, key=lambda r: r.harga_akhir, reverse=True)
    paling_mahal  = rows_by_price[0]
    paling_murah  = rows_by_price[-1]

    print("⚠️  Catatan: Data diambil pada hari Minggu (bursa IDX tutup).")
    print("   Harga yang tercatat merupakan harga penutupan sesi terakhir.")
    print("   Return = 0% untuk semua saham karena tidak ada sesi perdagangan.")
    print()
    print("📊 Ranking Harga Penutupan (referensi valuasi):")
    for i, row in enumerate(rows_by_price, 1):
        print(f"   {i}. {row.symbol:<6} → Rp {row.harga_akhir:>8,.0f}")
    print()
    print(f"💰 Saham dengan harga tertinggi : {paling_mahal.symbol} (Rp {paling_mahal.harga_akhir:,.0f})")
    print(f"💡 Saham dengan harga terendah  : {paling_murah.symbol} (Rp {paling_murah.harga_akhir:,.0f})")
    print()
    print("Kesimpulan:")
    print(f"  Pada periode pengamatan (hari libur/weekend), seluruh saham blue-chip IDX")
    print(f"  yang dipantau tidak mengalami pergerakan harga karena bursa efek Indonesia")
    print(f"  hanya beroperasi pada hari Senin–Jumat pukul 09.00–15.30 WIB.")
    print(f"  Data yang tersimpan di HDFS merupakan harga referensi penutupan terakhir,")
    print(f"  dengan {paling_mahal.symbol} sebagai saham paling premium (Rp {paling_mahal.harga_akhir:,.0f}) dan")
    print(f"  {paling_murah.symbol} sebagai saham dengan harga unit terendah (Rp {paling_murah.harga_akhir:,.0f}).")
else:
    rows_sorted = sorted(rows_return, key=lambda r: r.return_pct, reverse=True)
    best  = rows_sorted[0]
    worst = rows_sorted[-1]
    positif = [r for r in rows_return if r.return_pct > 0]
    negatif = [r for r in rows_return if r.return_pct < 0]
    print(f"🏆 Return terbaik  : {best.symbol}  ({best.return_pct:+.2f}%)")
    print(f"⚠️  Return terburuk : {worst.symbol}  ({worst.return_pct:+.2f}%)")
    print()
    print("Kesimpulan:")
    print(f"  {len(positif)} saham naik, {len(negatif)} saham turun.")
    print(f"  {best.symbol} memimpin dengan return {best.return_pct:+.2f}%.")

📈 INTERPRETASI: Return Per Saham
  ASII   | Rp    5,975 | +0.00% | ⚪ FLAT
  BBCA   | Rp    5,850 | +0.00% | ⚪ FLAT
  BMRI   | Rp    4,390 | +0.00% | ⚪ FLAT
  BBRI   | Rp    2,990 | +0.00% | ⚪ FLAT
  TLKM   | Rp    2,810 | +0.00% | ⚪ FLAT

⚠️  Catatan: Data diambil pada hari Minggu (bursa IDX tutup).
   Harga yang tercatat merupakan harga penutupan sesi terakhir.
   Return = 0% untuk semua saham karena tidak ada sesi perdagangan.

📊 Ranking Harga Penutupan (referensi valuasi):
   1. ASII   → Rp    5,975
   2. BBCA   → Rp    5,850
   3. BMRI   → Rp    4,390
   4. BBRI   → Rp    2,990
   5. TLKM   → Rp    2,810

💰 Saham dengan harga tertinggi : ASII (Rp 5,975)
💡 Saham dengan harga terendah  : TLKM (Rp 2,810)

Kesimpulan:
  Pada periode pengamatan (hari libur/weekend), seluruh saham blue-chip IDX
  yang dipantau tidak mengalami pergerakan harga karena bursa efek Indonesia
  hanya beroperasi pada hari Senin–Jumat pukul 09.00–15.30 WIB.
  Data yang tersimpan di HDFS merupakan harga referensi

---
## Analisis 2 — Volatilitas Intraday

> **Pertanyaan bisnis:** Saham mana yang paling fluktuatif dalam periode pengamatan?
>
> **Metode:** Hitung standar deviasi harga (stddev) per ticker. Karena data dikumpulkan saat weekend, stddev = 0 (tidak ada fluktuasi). Analisis ini tetap menampilkan statistik deskriptif harga sebagai profil valuasi tiap saham.

In [8]:
df_volatilitas = spark.sql("""
    SELECT
        symbol,
        COUNT(*)                                           AS jumlah_data_poin,
        ROUND(AVG(price), 2)                               AS harga_rata_rata,
        ROUND(MIN(price), 2)                               AS harga_terendah,
        ROUND(MAX(price), 2)                               AS harga_tertinggi,
        ROUND(MAX(price) - MIN(price), 2)                  AS range_harga,
        ROUND(STDDEV(price), 4)                            AS stddev_harga,
        ROUND(STDDEV(price) / AVG(price) * 100, 4)        AS cv_pct
    FROM saham
    GROUP BY symbol
    ORDER BY harga_rata_rata DESC
""")

df_volatilitas.show(truncate=False)
df_volatilitas.createOrReplaceTempView("hasil_volatilitas")
rows_vol = df_volatilitas.collect()

+------+----------------+---------------+--------------+---------------+-----------+------------+------+
|symbol|jumlah_data_poin|harga_rata_rata|harga_terendah|harga_tertinggi|range_harga|stddev_harga|cv_pct|
+------+----------------+---------------+--------------+---------------+-----------+------------+------+
|ASII  |75              |5975.0         |5975.0        |5975.0         |0.0        |0.0         |0.0   |
|BBCA  |75              |5850.0         |5850.0        |5850.0         |0.0        |0.0         |0.0   |
|BMRI  |75              |4390.0         |4390.0        |4390.0         |0.0        |0.0         |0.0   |
|BBRI  |75              |2990.0         |2990.0        |2990.0         |0.0        |0.0         |0.0   |
|TLKM  |75              |2810.0         |2810.0        |2810.0         |0.0        |0.0         |0.0   |
+------+----------------+---------------+--------------+---------------+-----------+------------+------+



In [9]:
print("=" * 70)
print("📊 INTERPRETASI: Volatilitas & Statistik Harga Intraday")
print("=" * 70)
print(f"{'Saham':<8} {'Data Poin':>10} {'Rata-rata':>14} {'Stddev':>12} {'CV%':>8}")
print("-" * 70)

for row in rows_vol:
    print(f"{row.symbol:<8} {row.jumlah_data_poin:>10,} "
          f"Rp {row.harga_rata_rata:>10,.0f} "
          f"{row.stddev_harga:>12.4f} "
          f"{row.cv_pct:>7.4f}%")

print()

if IS_FLAT:
    # Urutkan by harga untuk insight bermakna
    rows_vol_sorted = sorted(rows_vol, key=lambda r: r.harga_rata_rata, reverse=True)
    print("⚠️  Catatan: Stddev = 0 karena data dikumpulkan saat bursa tutup (weekend).")
    print("   Tidak ada fluktuasi harga intraday yang bisa dihitung.")
    print()
    print("📊 Profil Harga Penutupan per Saham:")

    # Hitung total data poin
    total_poin = sum(r.jumlah_data_poin for r in rows_vol)
    max_bar = 25
    max_poin = max(r.jumlah_data_poin for r in rows_vol)

    for row in rows_vol_sorted:
        bar = "█" * int((row.jumlah_data_poin / max_poin) * max_bar)
        print(f"   {row.symbol:<6} {bar:<25} {row.jumlah_data_poin:>4} data poin | "
              f"Harga: Rp {row.harga_rata_rata:>8,.0f}")

    print()
    print(f"   Total data poin terkumpul : {total_poin:,} record")
    print(f"   Rata-rata per saham       : {total_poin // len(rows_vol):,} record")
    print()
    print("Kesimpulan:")
    print(f"  Karena pengambilan data dilakukan pada hari libur bursa, seluruh saham")
    print(f"  menunjukkan volatilitas intraday = 0 (harga konstan). Sistem pipeline")
    print(f"  berhasil mengumpulkan total {total_poin:,} data poin dari 5 saham blue-chip IDX.")
    print(f"  Analisis volatilitas yang bermakna dapat dilakukan pada hari bursa aktif")
    print(f"  (Senin–Jumat) ketika harga bergerak sesuai kondisi pasar.")
else:
    most_vol  = rows_vol[0]
    least_vol = rows_vol[-1]
    print(f"🔥 Paling volatil : {most_vol.symbol} (stddev=Rp {most_vol.stddev_harga:,.2f}, CV={most_vol.cv_pct:.4f}%)")
    print(f"✅ Paling stabil  : {least_vol.symbol} (stddev=Rp {least_vol.stddev_harga:,.2f})")

📊 INTERPRETASI: Volatilitas & Statistik Harga Intraday
Saham     Data Poin      Rata-rata       Stddev      CV%
----------------------------------------------------------------------
ASII             75 Rp      5,975       0.0000  0.0000%
BBCA             75 Rp      5,850       0.0000  0.0000%
BMRI             75 Rp      4,390       0.0000  0.0000%
BBRI             75 Rp      2,990       0.0000  0.0000%
TLKM             75 Rp      2,810       0.0000  0.0000%

⚠️  Catatan: Stddev = 0 karena data dikumpulkan saat bursa tutup (weekend).
   Tidak ada fluktuasi harga intraday yang bisa dihitung.

📊 Profil Harga Penutupan per Saham:
   ASII   █████████████████████████   75 data poin | Harga: Rp    5,975
   BBCA   █████████████████████████   75 data poin | Harga: Rp    5,850
   BMRI   █████████████████████████   75 data poin | Harga: Rp    4,390
   BBRI   █████████████████████████   75 data poin | Harga: Rp    2,990
   TLKM   █████████████████████████   75 data poin | Harga: Rp    2,810

   T

---
## Analisis 3 — Frekuensi Sebutan Perusahaan di Berita RSS

> **Pertanyaan bisnis:** Perusahaan blue-chip mana yang paling banyak disebut di berita pasar modal?
>
> **Metode:** Pencarian keyword nama perusahaan (BCA, BRI, Telkom, Astra, Mandiri) di field `title` dan `summary_clean` menggunakan regex case-insensitive.

In [10]:
KEYWORD_MAP = {
    "BCA":     {"perusahaan": "Bank Central Asia",     "symbol": "BBCA"},
    "BRI":     {"perusahaan": "Bank Rakyat Indonesia",  "symbol": "BBRI"},
    "Telkom":  {"perusahaan": "Telkom Indonesia",       "symbol": "TLKM"},
    "Astra":   {"perusahaan": "Astra International",   "symbol": "ASII"},
    "Mandiri": {"perusahaan": "Bank Mandiri",           "symbol": "BMRI"},
}

# Tambahkan kolom mention untuk tiap keyword
df_mention = df_rss
for keyword in KEYWORD_MAP:
    pattern = f"(?i){keyword}"
    df_mention = df_mention.withColumn(
        f"mention_{keyword.lower()}",
        F.when(
            F.col("title").rlike(pattern) |
            F.col("summary_clean").rlike(pattern),
            1
        ).otherwise(0)
    )

# Hitung total per keyword
agg_exprs = [F.sum(f"mention_{kw.lower()}").alias(kw) for kw in KEYWORD_MAP]
summary_row = df_mention.agg(*agg_exprs).collect()[0]

mention_data = []
for keyword, info in KEYWORD_MAP.items():
    count = int(summary_row[keyword]) if summary_row[keyword] is not None else 0
    mention_data.append((keyword, info["perusahaan"], info["symbol"], count))

df_frekuensi = spark.createDataFrame(
    sorted(mention_data, key=lambda x: x[3], reverse=True),
    schema=["keyword", "perusahaan", "symbol", "jumlah_sebutan"]
)

df_frekuensi.show(truncate=False)
df_frekuensi.createOrReplaceTempView("hasil_frekuensi")
rows_freq = df_frekuensi.collect()

+-------+---------------------+------+--------------+
|keyword|perusahaan           |symbol|jumlah_sebutan|
+-------+---------------------+------+--------------+
|BCA    |Bank Central Asia    |BBCA  |0             |
|BRI    |Bank Rakyat Indonesia|BBRI  |0             |
|Telkom |Telkom Indonesia     |TLKM  |0             |
|Astra  |Astra International  |ASII  |0             |
|Mandiri|Bank Mandiri         |BMRI  |0             |
+-------+---------------------+------+--------------+



In [11]:
print("=" * 65)
print("📰 INTERPRETASI: Frekuensi Sebutan Perusahaan di Berita")
print("=" * 65)
print(f"Total artikel RSS yang dianalisis : {total_rss:,}")
print()

max_count = max(r.jumlah_sebutan for r in rows_freq) or 1
max_bar   = 30
total_sebutan = sum(r.jumlah_sebutan for r in rows_freq)

for row in rows_freq:
    bar_len = int((row.jumlah_sebutan / max_count) * max_bar)
    bar     = "█" * bar_len if bar_len > 0 else "░"
    pct     = row.jumlah_sebutan / total_rss * 100 if total_rss > 0 else 0
    print(f"  {row.keyword:<8} ({row.symbol})  {bar:<30}  {row.jumlah_sebutan:>3} artikel  ({pct:.1f}%)")

print()
top = rows_freq[0]

if total_sebutan == 0:
    print("⚠️  Tidak ada keyword perusahaan yang ditemukan dalam artikel RSS.")
    print()
    print("Analisis konten berita yang tersedia:")
    # Tampilkan judul artikel yang ada
    artikel = df_rss.select("title", "published").collect()
    for i, art in enumerate(artikel, 1):
        print(f"  {i}. {art.title}")
        print(f"     Terbit: {art.published}")
    print()
    print("Kesimpulan:")
    print(f"  Dari {total_rss} artikel yang berhasil dikumpulkan pada periode ini,")
    print(f"  tidak ada yang secara eksplisit menyebut nama perusahaan saham blue-chip")
    print(f"  (BCA, BRI, Telkom, Astra, Mandiri) dalam judul maupun ringkasan.")
    print(f"  Topik yang dominan adalah isu ketenagakerjaan dan teknologi AI.")
    print(f"  Ini menunjukkan bahwa pada hari tersebut tidak ada berita korporat signifikan")
    print(f"  yang berkaitan langsung dengan saham-saham yang dipantau.")
    print(f"  Analisis frekuensi yang lebih representatif membutuhkan akumulasi data")
    print(f"  dari beberapa hari perdagangan aktif.")
else:
    print(f"🏆 Paling banyak disebut : {top.perusahaan} ({top.symbol}) — {top.jumlah_sebutan} artikel")
    print()
    print("Kesimpulan:")
    print(f"  {top.perusahaan} ({top.symbol}) mendominasi pemberitaan dengan {top.jumlah_sebutan} sebutan.")
    print(f"  Tingginya frekuensi pemberitaan dapat mengindikasikan adanya katalis")
    print(f"  yang berpotensi mempengaruhi pergerakan harga saham {top.symbol}.")

📰 INTERPRETASI: Frekuensi Sebutan Perusahaan di Berita
Total artikel RSS yang dianalisis : 3

  BCA      (BBCA)  ░                                 0 artikel  (0.0%)
  BRI      (BBRI)  ░                                 0 artikel  (0.0%)
  Telkom   (TLKM)  ░                                 0 artikel  (0.0%)
  Astra    (ASII)  ░                                 0 artikel  (0.0%)
  Mandiri  (BMRI)  ░                                 0 artikel  (0.0%)

⚠️  Tidak ada keyword perusahaan yang ditemukan dalam artikel RSS.

Analisis konten berita yang tersedia:
  1. Disrupsi AI: Bagaimana AI Mengubah Lanskap Tenaga Kerja?
     Terbit: Sun, 03 May 2026 09:00:13 +0700
  2. Wajah Lain Pasar Kerja Global dan Tantangan Kualitas Pekerjaan
     Terbit: Sun, 03 May 2026 09:00:36 +0700
  3. Transmart Full Day Sale, Cek Aneka Produk yang Diskon Hari Ini
     Terbit: Sun, 03 May 2026 08:30:35 +0700

Kesimpulan:
  Dari 3 artikel yang berhasil dikumpulkan pada periode ini,
  tidak ada yang secara eksplisit men

---
## 4. Simpan Hasil ke HDFS

In [12]:
# Set HADOOP_USER_NAME = root sebelum write agar punya permission
os.environ["HADOOP_USER_NAME"] = "root"

def save_to_hdfs(df, hdfs_path, label):
    try:
        df.coalesce(1).write.mode("overwrite").json(hdfs_path)
        print(f"✅ [{label}] Berhasil disimpan ke HDFS: {hdfs_path}")
    except Exception as e:
        print(f"❌ [{label}] Gagal: {str(e)[:120]}")

save_to_hdfs(df_return,      f"{HDFS_HASIL_PATH}return/",      "Return Saham")
save_to_hdfs(df_volatilitas, f"{HDFS_HASIL_PATH}volatilitas/", "Volatilitas")
save_to_hdfs(df_frekuensi,   f"{HDFS_HASIL_PATH}frekuensi/",   "Frekuensi Berita")

# Verifikasi via Spark (tanpa subprocess hdfs)
print()
print("📁 Verifikasi isi HDFS /data/saham/hasil/:")
try:
    sc = spark.sparkContext
    hadoop = sc._jvm.org.apache.hadoop
    fs = hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
    path = hadoop.fs.Path("/data/saham/hasil/")
    status = fs.listStatus(path)
    for s in status:
        print(f"  {s.getPath()}  ({s.getLen()} bytes)")
except Exception as e:
    print(f"  (Tidak bisa list — cek manual di HDFS browser: http://localhost:9870)")

✅ [Return Saham] Berhasil disimpan ke HDFS: hdfs://namenode:8020/data/saham/hasil/return/
✅ [Volatilitas] Berhasil disimpan ke HDFS: hdfs://namenode:8020/data/saham/hasil/volatilitas/
✅ [Frekuensi Berita] Berhasil disimpan ke HDFS: hdfs://namenode:8020/data/saham/hasil/frekuensi/

📁 Verifikasi isi HDFS /data/saham/hasil/:
  hdfs://namenode:8020/data/saham/hasil/frekuensi  (0 bytes)
  hdfs://namenode:8020/data/saham/hasil/return  (0 bytes)
  hdfs://namenode:8020/data/saham/hasil/volatilitas  (0 bytes)


---
## 5. Simpan Ringkasan ke Dashboard (`spark_results.json`)

In [13]:
now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

return_records = [
    {
        "symbol":      row.symbol,
        "harga_awal":  row.harga_awal,
        "harga_akhir": row.harga_akhir,
        "return_pct":  row.return_pct,
        "selisih_rp":  row.selisih_rp,
        "waktu_awal":  str(row.waktu_awal),
        "waktu_akhir": str(row.waktu_akhir),
    }
    for row in rows_return
]

vol_records = [
    {
        "symbol":           row.symbol,
        "jumlah_data_poin": row.jumlah_data_poin,
        "harga_rata_rata":  row.harga_rata_rata,
        "harga_terendah":   row.harga_terendah,
        "harga_tertinggi":  row.harga_tertinggi,
        "range_harga":      row.range_harga,
        "stddev_harga":     float(row.stddev_harga) if row.stddev_harga else 0.0,
        "cv_pct":           float(row.cv_pct) if row.cv_pct else 0.0,
    }
    for row in rows_vol
]

freq_records = [
    {
        "keyword":        row.keyword,
        "perusahaan":     row.perusahaan,
        "symbol":         row.symbol,
        "jumlah_sebutan": row.jumlah_sebutan,
    }
    for row in rows_freq
]

spark_results = {
    "generated_at":          now_str,
    "is_market_open":        not IS_FLAT,
    "total_records_api":     total_api,
    "total_articles_rss":    total_rss,
    "analisis_return":       return_records,
    "analisis_volatilitas":  vol_records,
    "analisis_frekuensi":    freq_records,
}

with open(SPARK_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2)

print(f"✅ spark_results.json berhasil disimpan ke:")
print(f"   {SPARK_RESULTS_PATH}")
print()
# Preview
preview = json.dumps(spark_results, ensure_ascii=False, indent=2)
print(preview[:1000])
print("... (truncated)")

✅ spark_results.json berhasil disimpan ke:
   /home/jovyan/dashboard/data/spark_results.json

{
  "generated_at": "2026-05-03 03:45:37",
  "is_market_open": false,
  "total_records_api": 375,
  "total_articles_rss": 3,
  "analisis_return": [
    {
      "symbol": "ASII",
      "harga_awal": 5975.0,
      "harga_akhir": 5975.0,
      "return_pct": 0.0,
      "selisih_rp": 0.0,
      "waktu_awal": "2026-05-03 08:15:58.931700",
      "waktu_akhir": "2026-05-03 09:30:52.773903"
    },
    {
      "symbol": "BBCA",
      "harga_awal": 5850.0,
      "harga_akhir": 5850.0,
      "return_pct": 0.0,
      "selisih_rp": 0.0,
      "waktu_awal": "2026-05-03 08:15:58.531647",
      "waktu_akhir": "2026-05-03 09:30:52.385136"
    },
    {
      "symbol": "BMRI",
      "harga_awal": 4390.0,
      "harga_akhir": 4390.0,
      "return_pct": 0.0,
      "selisih_rp": 0.0,
      "waktu_awal": "2026-05-03 08:15:59.070103",
      "waktu_akhir": "2026-05-03 09:30:52.898169"
    },
    {
      "symbol": "BBR

---
## 6. Ringkasan Eksekutif

In [14]:
rows_by_price = sorted(rows_return, key=lambda r: r.harga_akhir, reverse=True)
rows_vol_sorted = sorted(rows_vol, key=lambda r: r.harga_rata_rata, reverse=True)

print("=" * 65)
print("🏦 RINGKASAN EKSEKUTIF — SahamMeter IDX")
print(f"   Waktu analisis : {now_str}")
print(f"   Status bursa   : {'🔴 TUTUP (weekend/libur)' if IS_FLAT else '🟢 BUKA'}")
print("=" * 65)
print()
print("📈 Analisis 1 — Posisi Harga Penutupan")
for row in rows_by_price:
    print(f"   {row.symbol:<6} → Rp {row.harga_akhir:>8,.0f}  (return: {row.return_pct:+.2f}%)")
print()
print("📊 Analisis 2 — Data Poin Terkumpul per Saham")
for row in rows_vol_sorted:
    print(f"   {row.symbol:<6} → {row.jumlah_data_poin:>5,} record  | avg Rp {row.harga_rata_rata:>8,.0f}")
print(f"   Total : {total_api:,} record dari HDFS")
print()
print("📰 Analisis 3 — Frekuensi Sebutan di Berita")
for row in rows_freq:
    print(f"   {row.keyword:<8} ({row.symbol}) → {row.jumlah_sebutan} sebutan")
print(f"   Total artikel RSS : {total_rss}")
print()
print("=" * 65)
print("✅ Semua analisis selesai!")
print(f"   HDFS  : {HDFS_HASIL_PATH}")
print(f"   Lokal : {SPARK_RESULTS_PATH}")
print("=" * 65)

🏦 RINGKASAN EKSEKUTIF — SahamMeter IDX
   Waktu analisis : 2026-05-03 03:45:37
   Status bursa   : 🔴 TUTUP (weekend/libur)

📈 Analisis 1 — Posisi Harga Penutupan
   ASII   → Rp    5,975  (return: +0.00%)
   BBCA   → Rp    5,850  (return: +0.00%)
   BMRI   → Rp    4,390  (return: +0.00%)
   BBRI   → Rp    2,990  (return: +0.00%)
   TLKM   → Rp    2,810  (return: +0.00%)

📊 Analisis 2 — Data Poin Terkumpul per Saham
   ASII   →    75 record  | avg Rp    5,975
   BBCA   →    75 record  | avg Rp    5,850
   BMRI   →    75 record  | avg Rp    4,390
   BBRI   →    75 record  | avg Rp    2,990
   TLKM   →    75 record  | avg Rp    2,810
   Total : 375 record dari HDFS

📰 Analisis 3 — Frekuensi Sebutan di Berita
   BCA      (BBCA) → 0 sebutan
   BRI      (BBRI) → 0 sebutan
   Telkom   (TLKM) → 0 sebutan
   Astra    (ASII) → 0 sebutan
   Mandiri  (BMRI) → 0 sebutan
   Total artikel RSS : 3

✅ Semua analisis selesai!
   HDFS  : hdfs://namenode:8020/data/saham/hasil/
   Lokal : /home/jovyan/dashb

In [15]:
spark.stop()
print("✅ SparkSession ditutup.")

✅ SparkSession ditutup.
